In [6]:
%%writefile run_variants.py
"""Robustness check: the same LightGBM pipeline on Base and Variants I-V.
Run:  python run_variants.py     (takes about 10 minutes)"""
import json
import time

import joblib
import numpy as np
import pandas as pd

import config
from src import data, evaluation, fairness, modelling

DESCRIPTIONS = {
    "Base": "Sampled to best represent the original dataset",
    "Variant I": "Higher group size disparity than Base",
    "Variant II": "Higher prevalence disparity than Base",
    "Variant III": "Better separability for one of the groups",
    "Variant IV": "Higher prevalence disparity in train",
    "Variant V": "Better separability in train for one of the groups",
}

params = json.loads((config.OUTPUT_DIR / "metrics.json").read_text()) if (config.OUTPUT_DIR / "metrics.json").exists() else {}
best = joblib.load(config.MODEL_DIR / "fraud_model.joblib").info["lightgbm_settings"]

rows = []
for name in DESCRIPTIONS:
    t = time.time()
    df = data.load_baf(config.DATA_DIR / f"{name}.csv")
    train, valid, test = data.split_by_month(df)
    scorer = modelling.fit_model("lightgbm", data.prepare(train), train[config.TARGET].values, best)
    sv, st_ = scorer.score(data.prepare(valid)), scorer.score(data.prepare(test))
    yv, yt = valid[config.TARGET].values, test[config.TARGET].values
    thr = evaluation.threshold_at_fpr(yv, sv, config.FPR_HIGH)
    rep = fairness.group_report(yt, st_, data.age_group(test[config.AGE]), thr)
    m = evaluation.full_metrics(yt, st_, thr)
    rows.append({"Dataset": name.replace(" ", "_"), "What is different": DESCRIPTIONS[name], "fraud_rate": df[config.TARGET].mean(),
                 "share_50_plus": float((df[config.AGE] >= 50).mean()),
                 "recall_at_5pct_fpr": m["recall_at_5pct_fpr"], "pr_auc": m["pr_auc"],
                 "FPR_under_50": float(rep.loc[rep.group == "Under 50", "FPR"].iloc[0]),
                 "FPR_50_plus": float(rep.loc[rep.group == "50 and over", "FPR"].iloc[0]),
                 "predictive_equality_ratio": fairness.predictive_equality_ratio(rep)})
    print(name, f"{time.time() - t:.0f}s", rows[-1], flush=True)
    del df, train, valid, test
pd.DataFrame(rows).to_csv(config.OUTPUT_DIR / "variants_summary.csv", index=False)
print("saved variants_summary.csv")

Overwriting run_variants.py


In [7]:
import joblib
import config

names = ["Base", "Variant I", "Variant II", "Variant III", "Variant IV", "Variant V"]
for n in names:
    p = config.DATA_DIR / f"{n}.csv"
    print(f"{n:12s} -> exists: {p.exists()}")

fm = joblib.load(config.MODEL_DIR / "fraud_model.joblib")
print("Loaded LightGBM settings:", fm.info["lightgbm_settings"])

Base         -> exists: True
Variant I    -> exists: True
Variant II   -> exists: True
Variant III  -> exists: True
Variant IV   -> exists: True
Variant V    -> exists: True
Loaded LightGBM settings: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 300, 'scale_pos_weight': 1}


In [8]:
!python run_variants.py

Base 41s {'Dataset': 'Base', 'What is different': 'Sampled to best represent the original dataset', 'fraud_rate': np.float64(0.011029), 'share_50_plus': 0.183013, 'recall_at_5pct_fpr': 0.5361362056984017, 'pr_auc': 0.18992281347221948, 'FPR_under_50': 0.03466337551596464, 'FPR_50_plus': 0.10805123513266239, 'predictive_equality_ratio': 0.32080499101565924}
Variant I 42s {'Dataset': 'Variant_I', 'What is different': 'Higher group size disparity than Base', 'fraud_rate': np.float64(0.011029), 'share_50_plus': 0.110699, 'recall_at_5pct_fpr': 0.5229325920778318, 'pr_auc': 0.18498610200543952, 'FPR_under_50': 0.042504027106593345, 'FPR_50_plus': 0.052800651524748894, 'predictive_equality_ratio': 0.8049905802141611}
Variant II 40s {'Dataset': 'Variant_II', 'What is different': 'Higher prevalence disparity than Base', 'fraud_rate': np.float64(0.011029), 'share_50_plus': 0.505847, 'recall_at_5pct_fpr': 0.5552466990965949, 'pr_auc': 0.21044480570674165, 'FPR_under_50': 0.010144137896874536, 'FP

In [4]:
from pathlib import Path
import pandas as pd

variants_path = Path("outputs/variants_summary.csv")
print("Exists:", variants_path.exists())

df_var = pd.read_csv(variants_path)
display(df_var)

Exists: True


,Dataset,What is different,fraud_rate,share_50_plus,recall_at_5pct_fpr,pr_auc,FPR_under_50,FPR_50_plus,predictive_equality_ratio
0,Base,Sampled to best represent the original dataset,0.011029,0.183013,0.536136,0.189923,0.034663,0.108051,0.320805
1,Variant_I,Higher group size disparity than Base,0.011029,0.110699,0.522933,0.184986,0.042504,0.052801,0.804991
2,Variant_II,Higher prevalence disparity than Base,0.011029,0.505847,0.555247,0.210445,0.010144,0.088598,0.114496
3,Variant_III,Better separability for one of the groups,0.011030,0.505861,0.490196,0.139072,0.045909,0.046918,0.978493
4,Variant_IV,Higher prevalence disparity in train,0.011030,0.505812,0.432881,0.110645,0.012777,0.087485,0.146053
5,Variant_V,Better separability in train for one of the gr...,0.011030,0.505846,0.508673,0.147107,0.046001,0.049170,0.935547


In [9]:
display(df_var[["Dataset", "What is different", "fraud_rate", "share_50_plus", "recall_at_5pct_fpr", "pr_auc", "predictive_equality_ratio"]])

print("Recall@5% FPR range:", round(df_var["recall_at_5pct_fpr"].min(), 3), "to", round(df_var["recall_at_5pct_fpr"].max(), 3))
print("Fairness ratio range:", round(df_var["predictive_equality_ratio"].min(), 3), "to", round(df_var["predictive_equality_ratio"].max(), 3))

,Dataset,What is different,fraud_rate,share_50_plus,recall_at_5pct_fpr,pr_auc,predictive_equality_ratio
0,Base,Sampled to best represent the original dataset,0.011029,0.183013,0.536136,0.189923,0.320805
1,Variant_I,Higher group size disparity than Base,0.011029,0.110699,0.522933,0.184986,0.804991
2,Variant_II,Higher prevalence disparity than Base,0.011029,0.505847,0.555247,0.210445,0.114496
3,Variant_III,Better separability for one of the groups,0.011030,0.505861,0.490196,0.139072,0.978493
4,Variant_IV,Higher prevalence disparity in train,0.011030,0.505812,0.432881,0.110645,0.146053
5,Variant_V,Better separability in train for one of the gr...,0.011030,0.505846,0.508673,0.147107,0.935547


Recall@5% FPR range: 0.433 to 0.555
Fairness ratio range: 0.114 to 0.978
